# 2D Chess Detection — Colab Training

End-to-end on Colab: mount Drive → clone repo → download piece styles → generate synthetic data → train YOLO11m + EfficientNetV2-S → run on a chess-book PDF.

**Before running:** `Runtime → Change runtime type → T4/L4/A100 GPU`.

All checkpoints and inference outputs are auto-saved to Google Drive at `MyDrive/2d-chess-detection-artifacts/` so you don't lose work when the runtime resets.

## 0. Sanity check — GPU

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

**Run this first.** A popup will ask you to authenticate. After mount, every weight and inference output written under `data/checkpoints` or `data/output` lands directly on Drive — no manual copy needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ARTIFACTS = pathlib.Path('/content/drive/MyDrive/2d-chess-detection-artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for sub in ('checkpoints', 'output', 'piece_sets'):
    (ARTIFACTS / sub).mkdir(exist_ok=True)
print('Drive ready at', ARTIFACTS)
!ls -la '{ARTIFACTS}'

## 2. Clone the repo and link Drive folders

Symlinks `data/checkpoints` and `data/output` to Drive so every file written during training/inference is persisted automatically. Synthetic datasets (`data/detector`, `data/classifier`) stay on local SSD for speed — they're cheap to regenerate.

If your repo is private set `GIT_TOKEN`; otherwise leave the default URL.

In [ ]:
REPO_URL = 'https://github.com/gsdt/2d-chess-detection.git'
BRANCH = 'main'

import os, subprocess, pathlib, shutil
os.chdir('/content')
if not pathlib.Path('2d-chess-detection').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL], check=True)
os.chdir('/content/2d-chess-detection')

# Symlink Drive-backed dirs into the project tree.
data_dir = pathlib.Path('/content/2d-chess-detection/data')
data_dir.mkdir(exist_ok=True)
for sub in ('checkpoints', 'output', 'piece_sets'):
    target = data_dir / sub
    drive_path = ARTIFACTS / sub
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)
    target.symlink_to(drive_path)
    print(f'  {target} -> {drive_path}')

!pwd && git log -1 --oneline

## 3. Install dependencies

In [ ]:
!pip install -q python-chess pypdfium2 cairosvg ultralytics tqdm

In [ ]:
import torch, chess, cv2, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('cv2', cv2.__version__, 'chess', chess.__version__, 'ultralytics', ultralytics.__version__)

## 4. Download Lichess piece styles (~37 sets, runs once)

These cache to Drive (`piece_sets/`) so you only download once across runtime resets.

In [ ]:
!python -m src.synth.lichess_pieces --out data/piece_sets

## 5. Generate synthetic datasets

Datasets stay on local SSD (regenerate ~5-8 min on a fresh runtime). Each board now picks a random piece style from `data/piece_sets`.

In [ ]:
DET_PAGES = 1500   # synthetic book pages for YOLO
CLF_BOARDS = 3000  # synthetic boards → ~64 crops each → ~192k crops

!python -m src.synth.gen_detector --pages {DET_PAGES} --out data/detector --piece-sets data/piece_sets
!python -m src.synth.gen_classifier --boards {CLF_BOARDS} --out data/classifier --piece-sets data/piece_sets

In [ ]:
# Peek a sample to verify augmentation + style variation looks right.
import glob
from IPython.display import Image, display
page = sorted(glob.glob('data/detector/images/train/*.jpg'))[0]
print(page)
display(Image(filename=page, width=600))

## 6. Train YOLO11m board detector

Weights save to `data/checkpoints/yolo_runs/board_detector/weights/` → mirrored to Drive automatically.

In [ ]:
YOLO_EPOCHS = 30
YOLO_IMGSZ = 768   # T4 (15GB) OOMs at 960 with YOLO11m; 768 is safe and still plenty
YOLO_BATCH = 8     # bump to 16 on imgsz=768 if A100/L4

!python -m src.train.train_yolo \
    --data data/detector/data.yaml \
    --epochs {YOLO_EPOCHS} --imgsz {YOLO_IMGSZ} --batch {YOLO_BATCH} \
    --device 0 --name board_detector

In [ ]:
# YOLO writes a results.png with PR / loss curves — show it.
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/checkpoints/yolo_runs/board_detector/*.png'))[:3]:
    print(p); display(Image(filename=p, width=700))

## 7. Train piece classifier

Best checkpoint saves to `data/checkpoints/classifier.pt` → Drive.

In [ ]:
CLF_EPOCHS = 10
CLF_BATCH = 256   # T4 16GB; reduce if OOM

!python -m src.train.train_classifier \
    --epochs {CLF_EPOCHS} --img-size 96 --batch-size {CLF_BATCH} --workers 4 \
    --data data/classifier --out data/checkpoints/classifier.pt

## 8. Upload the PDF and run end-to-end inference

Pick A (upload) or B (Drive). Inference outputs go to `data/output/<pdf_stem>/` → Drive.

In [ ]:
# A) Upload from laptop
from google.colab import files
uploaded = files.upload()
PDF_PATH = '/content/2d-chess-detection/' + next(iter(uploaded))
print('PDF:', PDF_PATH)

In [ ]:
# B) Or read from Drive (skip if you used A)
# PDF_PATH = '/content/drive/MyDrive/Step 2 Extra.pdf'

In [ ]:
PAGE_FROM = 1
PAGE_TO = 6   # exclusive

!python -m src.pipeline.run \
    --pdf "{PDF_PATH}" \
    --detector-weights data/checkpoints/yolo_runs/board_detector/weights/best.pt \
    --classifier-weights data/checkpoints/classifier.pt \
    --pages {PAGE_FROM} {PAGE_TO} \
    --device cuda \
    --out data/output

## 9. Inspect results

In [ ]:
import json, glob, pathlib
from IPython.display import Image, display

stem = pathlib.Path(PDF_PATH).stem
out_root = pathlib.Path('data/output') / stem
results = json.loads((out_root / 'results.json').read_text())
print(f'pages: {len(results)}  total boards: {sum(len(p["boards"]) for p in results)}')
for p in results[:2]:
    for b in p['boards']:
        print(f'page {p["page"]} board {b["index"]} conf={b["yolo_conf"]:.2f}  fen={b["fen"]}')

In [ ]:
# Side-by-side overlays + predicted-FEN renderings.
from IPython.display import display, Image
import glob, pathlib

for overlay in sorted(glob.glob(str(out_root / '*overlay.jpg')))[:2]:
    print(overlay)
    display(Image(filename=overlay, width=600))
for board in sorted(glob.glob(str(out_root / '*board_*[0-9].jpg')))[:6]:
    pred = board.replace('.jpg', '_pred.png')
    print(board, '→', pathlib.Path(pred).name)
    display(Image(filename=board, width=240))
    if pathlib.Path(pred).exists():
        display(Image(filename=pred, width=240))

## 10. Done — artifacts on Drive

Check `MyDrive/2d-chess-detection-artifacts/`:
- `checkpoints/yolo_runs/board_detector/weights/best.pt` — YOLO11m
- `checkpoints/classifier.pt` — piece classifier
- `output/<pdf_stem>/` — overlays, board crops, predicted FEN renderings, `results.json`
- `piece_sets/` — Lichess SVG cache

Next runtime, skip cells 4-7 (download + generate + train) — symlinks at cell 2 reattach the existing weights.